In [1]:
import numpy as np, torch, torch.nn.functional as F, torch.nn as nn
from torch_geometric.nn import GCNConv, GATConv
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_curve
import random, time, warnings
warnings.filterwarnings('ignore')

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

In [2]:
# Generate balanced trial pairs from speaker folder structure
# Standard practice — VoxCeleb trial list was generated the same way

def generate_trials(utterances, n_trials=10000, seed=42):
    random.seed(seed)
    spk2utts = {}
    for utt in utterances:
        spk = utt.split("/")[0]   # first folder = speaker_id
        spk2utts.setdefault(spk, []).append(utt)

    # Remove speakers with < 2 utterances
    spk2utts = {s:u for s,u in spk2utts.items() if len(u) >= 2}
    speakers  = list(spk2utts.keys())
    print(f"Speakers: {len(speakers)}  Total utts: {sum(len(v) for v in spk2utts.values())}")

    trials, labels = [], []

    # Positive pairs — same speaker
    while len([l for l in labels if l==1]) < n_trials // 2:
        spk = random.choice(speakers)
        u1, u2 = random.sample(spk2utts[spk], 2)
        trials.append((u1, u2)); labels.append(1)

    # Negative pairs — different speakers
    while len([l for l in labels if l==0]) < n_trials // 2:
        spk1, spk2 = random.sample(speakers, 2)
        u1 = random.choice(spk2utts[spk1])
        u2 = random.choice(spk2utts[spk2])
        trials.append((u1, u2)); labels.append(0)

    return trials, np.array(labels)

In [3]:
def compute_eer_and_min_dcf(scores, labels):
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr  = 1 - tpr
    eer  = fpr[np.nanargmin(np.abs(fnr - fpr))]
    dcf  = np.min(1.0*fnr*0.01 + 1.0*fpr*0.99)
    return eer, dcf

In [4]:
def build_edge_graph(X_np, K=40):
    sim = X_np @ X_np.T
    edge_index, edge_attr = [], []
    for i in range(len(X_np)):
        top = np.argpartition(sim[i], -(K+1))[-(K+1):]
        top = top[np.argsort(sim[i][top])[::-1]]
        top = top[top != i][:K]
        for rank, j in enumerate(top):
            ei, ej = X_np[i], X_np[int(j)]
            edge_index.append([i, int(j)])
            edge_attr.append([float(sim[i,j]),
                               float(np.linalg.norm(ei-ej)),
                               abs(float(np.linalg.norm(ei))-float(np.linalg.norm(ej))),
                               1.0/(rank+1)])
    return (torch.tensor(edge_index, dtype=torch.long).T,
            torch.tensor(edge_attr,  dtype=torch.float))

In [5]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, in_dim)
    def forward(self, x, edge_index, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

class GAT(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden, heads=1, concat=False, dropout=0.3)
        self.gat2 = GATConv(hidden, in_dim, heads=1, concat=False, dropout=0.3)
    def forward(self, x, edge_index, edge_attr=None):
        x = F.relu(self.gat1(x, edge_index))
        return self.gat2(x, edge_index)

class ScalarGateGNN(nn.Module):
    def __init__(self, in_dim, edge_dim=4):
        super().__init__()
        self.gate1 = nn.Sequential(nn.Linear(edge_dim,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
        self.gate2 = nn.Sequential(nn.Linear(edge_dim,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
        self.conv1 = GCNConv(in_dim, in_dim)
        self.conv2 = GCNConv(in_dim, in_dim)
        self.bn1   = nn.BatchNorm1d(in_dim)
    def forward(self, x, edge_index, edge_attr):
        w1 = self.gate1(edge_attr).squeeze()
        w2 = self.gate2(edge_attr).squeeze()
        x  = F.relu(self.bn1(self.conv1(x, edge_index, w1)))
        return self.conv2(x, edge_index, w2)

In [6]:
def contrastive_loss(Z, y, edge_index, margin=0.3):
    zi  = Z[edge_index[0]]; zj = Z[edge_index[1]]
    pos = (y[edge_index[0]] == y[edge_index[1]])
    sim = F.cosine_similarity(zi, zj)
    l_pos = (1-sim[pos]).mean()              if pos.sum()  > 0 else torch.tensor(0.)
    l_neg = F.relu(sim[~pos]-margin).mean()  if (~pos).sum()>0 else torch.tensor(0.)
    return l_pos + l_neg

In [7]:
def run_experiment(X_np, utterances, model_type='cosine', edge_dim=4, epochs=20):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    X_np   = normalize(X_np)
    trials, labels = generate_trials(utterances, n_trials=10000)
    utt2idx = {u:i for i,u in enumerate(utterances)}

    # Cosine baseline — no graph
    if model_type == 'cosine':
        X = torch.tensor(X_np, dtype=torch.float)
        scores = np.array([float(F.cosine_similarity(
            X[utt2idx[u1]].unsqueeze(0), X[utt2idx[u2]].unsqueeze(0)))
            for u1,u2 in trials])
        return compute_eer_and_min_dcf(scores, labels)

    # Graph models
    speakers = [u.split("/")[0] for u in utterances]
    spk2id   = {s:i for i,s in enumerate(sorted(set(speakers)))}
    y        = torch.tensor([spk2id[s] for s in speakers], dtype=torch.long)

    X  = torch.tensor(X_np, dtype=torch.float).to(device)
    ei, ea = build_edge_graph(X_np, K=40)
    ei = ei.to(device); ea = ea.to(device)

    in_dim = X_np.shape[1]
    if   model_type == 'gcn':     model = GCN(in_dim).to(device)
    elif model_type == 'gat':     model = GAT(in_dim).to(device)
    elif model_type == 'scalar':  model = ScalarGateGNN(in_dim, edge_dim).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train(); optimizer.zero_grad()
        if model_type == 'scalar':
            Z = F.normalize(model(X, ei, ea[:, :edge_dim]), dim=1)
        else:
            Z = F.normalize(model(X, ei), dim=1)
        loss = contrastive_loss(Z, y.to(device), ei)
        loss.backward(); optimizer.step()
        if (epoch+1) % 5 == 0:
            print(f"  epoch {epoch+1}/{epochs}  loss={loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        if model_type == 'scalar':
            Z = F.normalize(model(X, ei, ea[:, :edge_dim]), dim=1).cpu()
        else:
            Z = F.normalize(model(X, ei), dim=1).cpu()

    scores = np.array([float(torch.dot(Z[utt2idx[u1]], Z[utt2idx[u2]]))
                       for u1,u2 in trials])
    return compute_eer_and_min_dcf(scores, labels)

In [8]:
print("=" * 65)
print("LIBRISPEECH TEST-CLEAN — TDNN EMBEDDINGS")
print("=" * 65)

X_tdnn = np.load("../outputs/librispeech_tdnn/xvectors.npy")
utts   = list(np.load("../outputs/librispeech_tdnn/utterances.npy"))
print(f"Shape: {X_tdnn.shape}  Utterances: {len(utts)}")

configs = [
    ('cosine',  None, 'Cosine baseline'),
    ('gcn',     None, 'GCN binary edges'),
    ('gat',     None, 'GAT binary edges'),
    ('scalar',  2,    'ScalarGate-v2 [cosine+L2]'),
]

print(f"\n{'Method':<35} {'EER (%)':>8} {'MinDCF':>10}  time")
print("-" * 60)
tdnn_results = {}
for mtype, edim, label in configs:
    t0 = time.time()
    kwargs = {'edge_dim': edim} if edim else {}
    eer, dcf = run_experiment(X_tdnn, utts, model_type=mtype, **kwargs)
    elapsed = time.time()-t0
    print(f"{label:<35} {eer*100:>8.2f} {dcf:>10.5f}  ({elapsed/60:.1f}min)")
    tdnn_results[label] = (eer*100, dcf)

LIBRISPEECH TEST-CLEAN — TDNN EMBEDDINGS
Shape: (2620, 512)  Utterances: 2620

Method                               EER (%)     MinDCF  time
------------------------------------------------------------
Speakers: 40  Total utts: 2620
Cosine baseline                         7.08    0.00362  (0.0min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.3678
  epoch 10/20  loss=0.2009
  epoch 15/20  loss=0.1419
  epoch 20/20  loss=0.1160
GCN binary edges                        2.64    0.00099  (0.2min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.4969
  epoch 10/20  loss=0.3110
  epoch 15/20  loss=0.1744
  epoch 20/20  loss=0.1276
GAT binary edges                        2.82    0.00073  (0.2min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.1152
  epoch 10/20  loss=0.0551
  epoch 15/20  loss=0.0318
  epoch 20/20  loss=0.0224
ScalarGate-v2 [cosine+L2]               1.72    0.00048  (0.2min)


In [9]:
print("\n" + "=" * 65)
print("LIBRISPEECH TEST-CLEAN — ECAPA EMBEDDINGS")
print("=" * 65)

X_ecapa = np.load("../outputs/librispeech_ecapa/xvectors.npy")
utts_e  = list(np.load("../outputs/librispeech_ecapa/utterances.npy"))
print(f"Shape: {X_ecapa.shape}  Utterances: {len(utts_e)}")

print(f"\n{'Method':<35} {'EER (%)':>8} {'MinDCF':>10}  time")
print("-" * 60)
for mtype, edim, label in configs:
    t0 = time.time()
    kwargs = {'edge_dim': edim} if edim else {}
    eer, dcf = run_experiment(X_ecapa, utts_e, model_type=mtype, **kwargs)
    elapsed = time.time()-t0
    print(f"{label:<35} {eer*100:>8.2f} {dcf:>10.5f}  ({elapsed/60:.1f}min)")


LIBRISPEECH TEST-CLEAN — ECAPA EMBEDDINGS
Shape: (2620, 192)  Utterances: 2620

Method                               EER (%)     MinDCF  time
------------------------------------------------------------
Speakers: 40  Total utts: 2620
Cosine baseline                         1.60    0.00114  (0.0min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.1490
  epoch 10/20  loss=0.1043
  epoch 15/20  loss=0.0855
  epoch 20/20  loss=0.0483
GCN binary edges                        0.94    0.00045  (0.1min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.1639
  epoch 10/20  loss=0.1222
  epoch 15/20  loss=0.0972
  epoch 20/20  loss=0.0832
GAT binary edges                        1.54    0.00048  (0.1min)
Speakers: 40  Total utts: 2620
  epoch 5/20  loss=0.1323
  epoch 10/20  loss=0.0916
  epoch 15/20  loss=0.0575
  epoch 20/20  loss=0.0320
ScalarGate-v2 [cosine+L2]               1.00    0.00035  (0.1min)
